In [ ]:
import os
import astropy_iers_data as aid

for name in ("IERS_A_FILE", "IERS_B_FILE", "IERS_LEAP_SECOND_FILE"):
    path = str(getattr(aid, name))
    size = os.path.getsize(path)
    print(f"{name}: {os.path.basename(path)} ({size:,} bytes)")
    assert os.path.isfile(path), f"{name} missing on disk"

assert os.path.getsize(str(aid.IERS_A_FILE)) > 1_000_000   # finals2000A.all ~3.7 MB
assert os.path.getsize(str(aid.IERS_B_FILE)) > 1_000_000   # eopc04.1962-now ~5 MB
assert os.path.getsize(str(aid.IERS_LEAP_SECOND_FILE)) > 500
print("all IERS data files present")

In [ ]:
# Example 2: parse the IERS-B (EOP 20 C04) table -- the primary data product.
import astropy_iers_data as aid

with open(str(aid.IERS_B_FILE)) as f:
    rows = [line.split() for line in f if line.strip() and not line.startswith("#")]

print(f"IERS-B data rows: {len(rows)}")
assert len(rows) > 20_000  # daily samples since 1962

# Columns: yr mon day hh MJD x_pole y_pole UT1-UTC ...
first_mjd, last_mjd = float(rows[0][4]), float(rows[-1][4])
print(f"MJD range: {first_mjd:.2f} .. {last_mjd:.2f}")
assert first_mjd < 38000  # series starts in 1962
assert last_mjd > 60000   # extends well past 2023

x_pole, y_pole = float(rows[-1][5]), float(rows[-1][6])
print(f"latest polar motion: x={x_pole} arcsec, y={y_pole} arcsec")
assert abs(x_pole) < 1.0 and abs(y_pole) < 1.0  # polar motion stays sub-arcsecond


In [ ]:
# Example 3: read the IERS-A (finals2000A.all) fixed-width records.
import astropy_iers_data as aid

with open(str(aid.IERS_A_FILE)) as f:
    lines = f.read().splitlines()

print(f"IERS-A records: {len(lines)}")
assert len(lines) > 18_000  # daily records since 1973

# Fixed-width layout: cols 8-15 MJD, col 17 I/P flag, cols 19-27 PM-x (arcsec).
first = lines[0]
mjd = float(first[7:15])
ip_flag = first[16]
pm_x = float(first[18:27])
print(f"first record: MJD={mjd}, flag={ip_flag!r}, PM-x={pm_x} arcsec")
assert mjd == 41684.0          # 1973-01-02, the series start
assert ip_flag in ("I", "P")   # IERS-determined vs predicted
assert abs(pm_x) < 1.0


In [ ]:
# Example 4: leap-second table invariant -- TAI-UTC is monotonically increasing.
import astropy_iers_data as aid

entries = []
with open(str(aid.IERS_LEAP_SECOND_FILE)) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            mjd, day, month, year, tai_utc = line.split()
            entries.append((float(mjd), int(tai_utc)))

print(f"leap-second entries: {len(entries)}")
print(f"first: {entries[0]}, last: {entries[-1]}")
assert len(entries) >= 28
assert entries[0][1] == 10          # 1972: TAI-UTC starts at 10 s
assert entries[-1][1] == 37         # no leap second introduced since 2017
mjds = [e[0] for e in entries]
offsets = [e[1] for e in entries]
assert mjds == sorted(mjds)
assert all(b - a == 1 for a, b in zip(offsets, offsets[1:]))  # each leap adds 1 s


In [ ]:
# Example 5: download-source constants + bundled READMEs (consumed by astropy.utils.iers).
import os
import astropy_iers_data as aid

for name in ("IERS_A_URL", "IERS_A_URL_MIRROR", "IERS_B_URL", "IERS_LEAP_SECOND_URL"):
    url = getattr(aid, name)
    print(f"{name} = {url}")
    assert url.startswith(("https://", "ftp://")), f"{name} not a fetchable URL"

for name in ("IERS_A_README", "IERS_B_README"):
    path = str(getattr(aid, name))
    assert os.path.isfile(path) and os.path.getsize(path) > 0
    print(f"{name}: {os.path.basename(path)} present")

data_dir = os.path.dirname(str(aid.IERS_A_FILE))
files = sorted(os.listdir(data_dir))
print(f"data dir contents: {files}")
assert len(files) >= 5
